# E3 — Batch: Batch Size en Llama-3.2-1B

## Pregunta de investigación

¿Cómo afecta el batch size al throughput y TTFT?

## Hipótesis previa

Mayor batch_size puede mejorar throughput pero no debería afectar TTFT individual.

## Configuración del experimento

| Parámetro | Valores |
|-----------|---------|
| Modelo | Llama-3.2-1B |
| Batch Size | 1, 2, 4, 8, 16 |
| Total | Múltiples runs |

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from monitorviz.io import load_collection
from monitorviz.viz import setup_style

setup_style("talk")
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Carga de datos (E3) ---
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
DATA_ROOT = PROJECT_ROOT / "data" / "tfg-data" / "E3"

assert DATA_ROOT.is_dir(), f"❌ No encontrado: {DATA_ROOT.resolve()}"

coll = load_collection(DATA_ROOT)
print(f"Runs cargados E3: {len(coll)}")

# --- Filtrado para 09 ---
summary = coll.summary_df()
hw_full = coll.hw_metrics_df()
pm_full = coll.prompt_metrics_df()

e3 = summary[
    (summary["test_type"] == "TYPE_3") &
    (summary["model_label"].str.contains("Llama-3.2-1B")) &
    (summary["accelerator"] == False)
].copy()

ex = e3

print(f"\nRuns en 09_E3_batch: {ex.shape[0]}")
if len(ex) > 0:
    print(f"Configuraciones únicas: {ex.shape[0]} runs")
else:
    print("Sin datos disponibles para este experimento")

## Resumen ejecutivo

> TODO: Actualizar con resultados finales.

Placeholder: 3-5 frases resumiendo los hallazgos principales.

## Comparativa de métricas de inferencia

Throughput, latencia, TTFT, perplejidad por configuración.

In [ ]:
# Tabla resumen
display_cols = [
    "model_label", "engine",
    "tokens_per_s_mean", "words_per_s_mean",
    "latency_ms_mean", "ttft_ms_mean",
    "perplexity_geomean",
]
cols = [c for c in display_cols if c in ex.columns]
if not ex.empty:
    display(ex[cols].round(3))
else:
    print("Sin datos para esta tabla")

## Comparativa de hardware

Temperatura, potencia, throttling.

In [ ]:
# Tabla hardware
if "temp_max_c" in ex.columns:
    hw_cols = [
        "model_label",
        "temp_max_c", "temp_mean_c",
        "power_mean_w", "power_max_w",
        "throttled_ratio",
    ]
    hw_cols = [c for c in hw_cols if c in ex.columns]
    if not ex.empty:
        display(ex[hw_cols].round(2))
else:
    print("Sin datos de hardware")

## Timelines de temperatura y potencia

Small multiples con un subplot por modelo.

In [ ]:

# Timelines de temperatura
hw_ex = hw_full[hw_full["run_id"].isin(ex["run_id"])]

if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    if len(models) == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nSin datos", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        sns.lineplot(data=hw_m, x="t_rel_s", y="temperature_c", ax=ax, alpha=0.7)
        ax.axhline(80, ls="--", color="red", alpha=0.5, label="80°C")
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title(f"{model}", fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for ax in axes[len(models):]:
        ax.set_visible(False)

    fig.suptitle("Temperatura por modelo", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware")


## Distribución CPU y Memoria

In [ ]:
# Boxplot CPU y Memoria
hw_ex = hw_full[hw_full["run_id"].isin(ex["run_id"])]
if not hw_ex.empty:
    hw_inference = hw_ex[hw_ex["cpu_usage_pct"] > 50]
    fig, axes = plt.subplots(2, 1, figsize=(12, 9))
    order = hw_ex.groupby("model_label")["cpu_usage_pct"].median().sort_values(ascending=False).index

    sns.boxplot(data=hw_inference, x="model_label", y="cpu_usage_pct", order=order, showfliers=False, ax=axes[0])
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    axes[0].set_ylabel("CPU (%)")
    axes[0].set_title("CPU durante inferencia")

    sns.boxplot(data=hw_ex, x="model_label", y="mem_pct", order=order, showfliers=False, ax=axes[1])
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    axes[1].set_ylabel("Memoria (%)")
    axes[1].set_title("Uso de memoria")
    fig.tight_layout()
    plt.show()
else:
    print("Sin datos de hardware")

## Análisis detallado (por run representativo)

In [ ]:
# Select median run and analyze
if not hw_ex.empty and len(ex) > 0:
    from monitorviz.viz import temp_freq_dual, cpu_memory_dual_phases, hw_distributions_panel

    run_dur = hw_ex.groupby("run_id")["t_rel_s"].max()
    if len(run_dur) > 0:
        med_run_id = run_dur.iloc[len(run_dur)//2]
        med_run = next((r for r in coll.runs if r.run_id == med_run_id), None)

        if med_run:
            hw_r = hw_ex[hw_ex["run_id"] == med_run_id]
            try:
                fig = temp_freq_dual(hw_r, med_run)
                plt.show()
            except:
                pass
else:
    print("Sin datos para análisis detallado")

## Sector específico del experimento

In [ ]:
# E3-specific: Batch size sensitivity
try:
    from monitorviz.viz import sensitivity_curve

    e3_ok = (
        not e3.empty
        and "batch_size" in e3.columns
        and e3["batch_size"].nunique() >= 2
    )

    if e3_ok:
        fig = sensitivity_curve(
            e3,
            param_col="batch_size",
            metrics=[
                ("ttft_ms_mean", "TTFT (ms)"),
                ("tokens_per_s_mean", "Throughput (tok/s)"),
            ],
            log_x=True
        )
        plt.show()

        print("ℹ️  Nota: El throughput de decode no debería variar significativamente con batch_size.")
    else:
        print("Sin datos suficientes de batch_size en E3")
except Exception as e:
    print(f"Error en análisis E3: {e}")

## Métricas de inferencia (histogramas)

In [ ]:
# Inference metrics
def _plot_metric(df, metric, ylabel, title, unit_label=""):
    data = df.dropna(subset=[metric]).copy()
    if data.empty:
        print(f"Sin datos para {metric}")
        return
    fig, ax = plt.subplots(figsize=(10, 6))
    order = data.groupby("model_label")[metric].mean().sort_values().index
    sns.barplot(data=data, x="model_label", y=metric, order=order, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

pm_valid = pm_full[(pm_full["run_id"].isin(ex["run_id"])) & (~pm_full["is_empty_generation"]) & (pm_full["latency_ms"] > 0)]
if not pm_valid.empty:
    _plot_metric(pm_valid, "tokens_per_second", "tokens/s", "Throughput")
else:
    print("Sin datos de inferencia")

## Alertas y anomalías

In [ ]:
# Alerts
alerts = []
throttled = ex[ex["throttled_ratio"] > 0.5]
if not throttled.empty:
    for _, row in throttled.iterrows():
        alerts.append(f"⚠️  {row['model_label']}: {row['throttled_ratio']*100:.1f}% throttling")
if alerts:
    for a in alerts:
        print(a)
else:
    print("✅ Sin alertas detectadas")

## Timelines ampliados

Análisis temporal completo de temperatura, potencia, CPU y swap.

In [ ]:
# Power timeline (complete section)
if not hw_ex.empty:
    models = sorted(ex["model_label"].unique())
    ncols = 2
    nrows = (len(models) + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    if len(models) == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, model in zip(axes, models):
        hw_m = hw_ex[hw_ex["model_label"] == model]
        if hw_m.empty:
            ax.text(0.5, 0.5, f"{model}\nSin datos", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        sns.lineplot(data=hw_m, x="t_rel_s", y="internal_power_w", ax=ax, alpha=0.7)
        total_s = hw_m["t_rel_s"].max()
        ax.set_xlabel("Tiempo (min)" if total_s > 1800 else "Tiempo (s)")
        ax.set_ylabel("Potencia (W)")
        ax.set_title(f"{model}", fontweight="bold")
        ax.grid(True, alpha=0.3)

    for ax in axes[len(models):]:
        ax.set_visible(False)

    fig.suptitle("Potencia interna por modelo (ampliado)", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()

## Distribución detallada CPU y Memoria

Violin plots con strip para mejor visualización.

In [ ]:
# Violin + strip version (with FIX rotation=45)
if not hw_ex.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    active = hw_ex[hw_ex["cpu_usage_pct"] > 50]
    order_cpu = active.groupby("model_label")["cpu_usage_pct"].median().sort_values(ascending=False).index if not active.empty else []

    if len(order_cpu) > 0:
        sns.violinplot(data=active, x="model_label", y="cpu_usage_pct", order=order_cpu, inner=None, alpha=0.6, ax=axes[0])
        sns.stripplot(data=active, x="model_label", y="cpu_usage_pct", order=order_cpu, size=2, alpha=0.3, color="black", ax=axes[0])
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
        axes[0].set_title("CPU durante inferencia (violin)")
        axes[0].set_ylabel("CPU (%)")
        axes[0].set_ylim(80, 102)

    order_mem = hw_ex.groupby("model_label")["mem_pct"].median().sort_values(ascending=False).index
    sns.violinplot(data=hw_ex, x="model_label", y="mem_pct", order=order_mem, inner=None, alpha=0.6, ax=axes[1])
    sns.stripplot(data=hw_ex, x="model_label", y="mem_pct", order=order_mem, size=2, alpha=0.3, color="black", ax=axes[1])
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
    axes[1].set_title("Memoria (violin)")
    axes[1].set_ylabel("Memoria (%)")

    fig.tight_layout()
    plt.show()

## Análisis detallado con funciones compostas

Visualizaciones complejas para un run representativo.

In [ ]:
# Use composite plotting functions
try:
    from monitorviz.viz import temp_freq_dual, cpu_memory_dual_phases, hw_distributions_panel

    if not hw_ex.empty and len(ex) > 0:
        run_durations = hw_ex.groupby("run_id")["t_rel_s"].max()
        if len(run_durations) > 0:
            median_idx = len(run_durations) // 2
            representative_run_id = run_durations.iloc[median_idx]

            representative_run = None
            for r in coll.runs:
                if r.run_id == representative_run_id:
                    representative_run = r
                    break

            if representative_run:
                hw_r = hw_ex[hw_ex["run_id"] == representative_run_id]

                try:
                    fig = temp_freq_dual(hw_r, representative_run)
                    plt.show()
                except Exception as e:
                    print(f"No se pudo generar temp_freq_dual: {e}")

                try:
                    fig = cpu_memory_dual_phases(hw_r, representative_run)
                    plt.show()
                except Exception as e:
                    print(f"No se pudo generar cpu_memory_dual_phases: {e}")
except Exception as e:
    print(f"Error en análisis detallado: {e}")

## Análisis energético completo

Energía por token, MBU, trade-offs Pareto.

In [ ]:
# Energy and efficiency analysis
if "energy_per_token_j" in ex.columns:
    energy_data = ex.dropna(subset=["energy_per_token_j"])
    if not energy_data.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        order = energy_data.groupby("model_label")["energy_per_token_j"].mean().sort_values().index
        sns.barplot(data=energy_data, x="model_label", y="energy_per_token_j", order=order, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_ylabel("Energía/token (J)")
        ax.set_title("Eficiencia energética por modelo")
        fig.tight_layout()
        plt.show()

# MBU analysis
if "mbu_pct" in ex.columns:
    mbu_data = ex.dropna(subset=["mbu_pct"])
    if not mbu_data.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        order = mbu_data.groupby("model_label")["mbu_pct"].mean().sort_values(ascending=False).index
        sns.barplot(data=mbu_data, x="model_label", y="mbu_pct", order=order, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_ylabel("MBU (%)")
        ax.set_title("Memory Bandwidth Utilization")
        fig.tight_layout()
        plt.show()

## Análisis multidimensional

Diagramas polares y correlación de Pearson.

In [ ]:
# Radar charts and correlation
try:
    from monitorviz.viz import correlation_heatmap

    if len(ex) >= 3:
        fig = correlation_heatmap(ex, title=f"Correlación Pearson (n={len(ex)} runs)")
        plt.show()
except Exception as e:
    print(f"Error en correlación: {e}")

## Conclusiones

> TODO: Actualizar con resultados finales.